In [0]:
# ==========================================================
# 01. EXPLORACIÓN DEL CATÁLOGO DE PRODUCTOS
# Proyecto: ElectroCasa
# ==========================================================

RUTA_CATALOGO = (
    "/Volumes/proyecto_final/default/vol_landing/catalogo"
)

ARCHIVO = (
    f"{RUTA_CATALOGO}/catalogo_productos_2026-03-01.json"
)

# Leer el archivo JSON
df_catalogo = (
    spark.read
    .option("multiLine", "true")
    .json(ARCHIVO)
)

print("ESQUEMA DEL CATÁLOGO:")
df_catalogo.printSchema()

print("TOTAL DE REGISTROS:")
print(df_catalogo.count())

display(df_catalogo.limit(10))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS proyecto_final.bronze.catalogo (
    producto_id STRING,
    nombre_producto STRING,
    categoria STRING,
    marca STRING,
    precio_lista STRING,
    activo BOOLEAN,
    archivo_origen STRING,
    fecha_ingesta TIMESTAMP
)
USING DELTA;

In [0]:
%sql
COPY INTO proyecto_final.bronze.catalogo
FROM (
    SELECT
        producto_id,
        nombre_producto,
        categoria,
        marca,
        precio_lista,
        activo,
        _metadata.file_name AS archivo_origen,
        current_timestamp() AS fecha_ingesta
    FROM '/Volumes/proyecto_final/default/vol_landing/catalogo'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true');

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.catalogo
GROUP BY archivo_origen
ORDER BY archivo_origen;